STEP 1 — Prepare Input File (URL List)
links_for_golbal_military_data.txt


STEP 2 — Import Required Libraries

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re


Step 3 : Final code


In [ ]:
# ======================================================
# Project: Unified Military Analytics & Comparison Dashboard
# Author : Rutuja Ghodake
# Description:
# Collects global military data from GlobalFirepower.com
# and merges all metrics into a single structured dataset.
# ======================================================

import requests                      # To send HTTP requests
from bs4 import BeautifulSoup        # For parsing HTML content
import pandas as pd                  # For data manipulation
import re                            # For cleaning numeric values

# ------------------------------
# Global Headers
# ------------------------------
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# -------------------------------------------------------
# Read URLs from text file
# -------------------------------------------------------
def read_links_txt(path):
    links = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("http"):
                links.append(line)
    return links


# -------------------------------------------------------
# Main Scraping Function
# -------------------------------------------------------
def scrape_global_firepower():

    base_url = "https://www.globalfirepower.com/countries-listing.php"
    metric_urls = read_links_txt("links_for_golbal_military_data.txt")

    # -------------------------------
    # STEP 1: Fetch country list
    # -------------------------------
    response = requests.get(base_url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    containers = soup.select("div.picTrans.recordsetContainer")

    countries = []
    ranks = []

    for box in containers:
        try:
            country = box.find("span", class_="textWhite textLarge textShadow").text.strip()
            rank = box.find("span", class_="textWhite textLarge textBold").text.strip()
            countries.append(country)
            ranks.append(rank)
        except:
            continue

    df = pd.DataFrame({
        "Country": countries,
        "Rank": ranks
    })

    # -------------------------------
    # STEP 2: Extract all metrics
    # -------------------------------
    for url in metric_urls:
        print(f"Scraping: {url}")

        response = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(response.text, "html.parser")

        rows = soup.select("div.picTrans.recordsetContainer")

        temp_countries = []
        values = []

        for row in rows:
            try:
                country = row.find("span", class_="textWhite textLarge textShadow").text.strip()
                value = row.find_all("span", class_="textWhite textLarge")[-1].text.strip()
                temp_countries.append(country)
                values.append(value)
            except:
                continue

        column_name = url.split("/")[-1].replace(".php", "")

        temp_df = pd.DataFrame({
            "Country": temp_countries,
            column_name: values
        })

        df = df.merge(temp_df, on="Country", how="left")

    # -------------------------------
    # STEP 3: Clean numeric values
    # -------------------------------
    for col in df.columns[2:]:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.extract(r"(\d+\.?\d*)")[0]
        )

    return df


# -------------------------------------------------------
# RUN SCRIPT
# -------------------------------------------------------
df = scrape_global_firepower()

# Save output
df.to_csv("global_military_data_final.csv", index=False)

# Display results
print("\n✅ Data extraction completed successfully!")
print("Total Countries:", len(df))
print("Total Features:", len(df.columns))
print("\nPreview:\n")
print(df.head())




Scraping: https://www.globalfirepower.com/total-population-by-country.php
Scraping: https://www.globalfirepower.com/available-military-manpower.php
Scraping: https://www.globalfirepower.com/manpower-fit-for-military-service.php
Scraping: https://www.globalfirepower.com/manpower-reaching-military-age-annually.php
Scraping: https://www.globalfirepower.com/active-military-manpower.php
Scraping: https://www.globalfirepower.com/active-reserve-military-manpower.php
Scraping: https://www.globalfirepower.com/manpower-paramilitary.php
Scraping: https://www.globalfirepower.com/capital-cities-by-total-population.php
Scraping: https://www.globalfirepower.com/aircraft-total.php
Scraping: https://www.globalfirepower.com/aircraft-total-fighters.php
Scraping: https://www.globalfirepower.com/aircraft-total-attack-types.php
Scraping: https://www.globalfirepower.com/aircraft-total-transports.php
Scraping: https://www.globalfirepower.com/aircraft-total-trainers.php
Scraping: https://www.globalfirepower.co